# Task 1: Install required Python libraries

In [ ]:
%pip install -q pandas scikit-learn streamlit joblib requests

import pandas
import sklearn
import streamlit
import joblib
import requests

print(pandas.__version__)
print(sklearn.__version__)
print(streamlit.__version__)
print(joblib.__version__)
print(requests.__version__)

# Task 2: Download the dataset

In [ ]:
from pathlib import Path
import requests
import pandas as pd

Path("data").mkdir(exist_ok=True)

csv_path = Path("data/results.csv")
if not csv_path.exists():
    url = "https://raw.githubusercontent.com/IBM-SkillsBuild-AI-Builders-Challenge/hands-on-labs/main/02_football_lab_june/04_data/results.csv"
    response = requests.get(url)
    csv_path.write_bytes(response.content)

matches = pd.read_csv("data/results.csv", parse_dates=["date"])

print(matches.shape)
print(matches["date"].min())
print(matches["date"].max())
matches.head(3)

# Task 3: Explore the data

In [ ]:
print(matches["tournament"].value_counts().head(10))

home_counts = matches["home_team"].value_counts()
away_counts = matches["away_team"].value_counts()
total_counts = home_counts.add(away_counts, fill_value=0).astype(int)
print(total_counts.sort_values(ascending=False).head(15))

matches["decade"] = (matches["date"].dt.year // 10 * 10).astype(str) + "s"
decade_counts = matches["decade"].value_counts().sort_index()
print(decade_counts)

# Task 4:Engineer features for the model

In [ ]:
def winrate(hist):
    if len(hist) == 0:
        return 0.5
    return sum(w for _, _, w in hist) / len(hist)

def goal_avg(hist):
    if len(hist) == 0:
        return 1.0
    return sum(gf for gf, _, _ in hist) / len(hist)

def recent_form(hist):
    if len(hist) < 10:
        return 0.5
    recent = hist[-10:]
    return sum(w for _, _, w in recent) / 10

filtered = matches[matches["date"] >= "1990-01-01"].sort_values("date").reset_index(drop=True)

team_history = {}
rows = []

major_tournaments = {"Soccer World Cup", "Soccer World Cup qualification", "UEFA Euro", "UEFA Euro qualification", "Copa América", "African Cup of Nations"}

for _, row in filtered.iterrows():
    home = row["home_team"]
    away = row["away_team"]
    
    if home not in team_history:
        team_history[home] = []
    if away not in team_history:
        team_history[away] = []
    
    home_hist = team_history[home]
    away_hist = team_history[away]
    
    team_a_winrate = winrate(home_hist)
    team_b_winrate = winrate(away_hist)
    team_a_goal_avg = goal_avg(home_hist)
    team_b_goal_avg = goal_avg(away_hist)
    team_a_recent_form = recent_form(home_hist)
    team_b_recent_form = recent_form(away_hist)
    is_neutral = int(row["neutral"])
    is_major_tournament = 1 if row["tournament"] in major_tournaments else 0
    
    home_score = row["home_score"]
    away_score = row["away_score"]
    
    if home_score > away_score:
        outcome = 0
    elif home_score == away_score:
        outcome = 1
    else:
        outcome = 2
    
    rows.append({
        "date": row["date"],
        "home_team": home,
        "away_team": away,
        "team_a_winrate": team_a_winrate,
        "team_b_winrate": team_b_winrate,
        "team_a_goal_avg": team_a_goal_avg,
        "team_b_goal_avg": team_b_goal_avg,
        "team_a_recent_form": team_a_recent_form,
        "team_b_recent_form": team_b_recent_form,
        "is_neutral": is_neutral,
        "is_major_tournament": is_major_tournament,
        "outcome": outcome
    })
    
    home_won = 1 if home_score > away_score else 0
    away_won = 1 if away_score > home_score else 0
    
    team_history[home].append((home_score, away_score, home_won))
    team_history[away].append((away_score, home_score, away_won))

features_df = pd.DataFrame(rows)

print(features_df.shape)
features_df.head(3)

# Task 5: Split data into training and test sets

In [ ]:
feature_cols = ["team_a_winrate", "team_b_winrate", "team_a_goal_avg", "team_b_goal_avg", "team_a_recent_form", "team_b_recent_form", "is_neutral", "is_major_tournament"]

train_df = features_df[features_df["date"] < "2018-01-01"]
test_df = features_df[features_df["date"] >= "2018-01-01"]

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_train = train_df["outcome"]
y_test = test_df["outcome"]

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)
print(y_train.value_counts(normalize=True))

# Task 6: Train and evaluate the model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
print(f"{test_accuracy * 100:.2f}%")

most_frequent_class = y_train.value_counts().idxmax()
baseline_accuracy = (y_test == most_frequent_class).mean()
print(f"{baseline_accuracy * 100:.2f}%")

cm = confusion_matrix(y_test, y_pred)
labels = ["Home win", "Draw", "Away win"]
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)

importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
for feature, importance in importances.items():
    print(f"{feature}: {importance}")

# Task 7: Save the model and team statistics

In [ ]:
from pathlib import Path
import joblib

Path("models").mkdir(exist_ok=True)

wc_qual = matches[matches["tournament"] == "Soccer World Cup qualification"]
soccer_teams = set(wc_qual["home_team"]) | set(wc_qual["away_team"])

team_stats = {}

for team in soccer_teams:
    home_matches = matches[matches["home_team"] == team]
    away_matches = matches[matches["away_team"] == team]
    
    total_matches = len(home_matches) + len(away_matches)
    
    if total_matches < 30:
        continue
    
    home_wins = (home_matches["home_score"] > home_matches["away_score"]).sum()
    away_wins = (away_matches["away_score"] > away_matches["home_score"]).sum()
    total_wins = home_wins + away_wins
    
    total_goals = home_matches["home_score"].sum() + away_matches["away_score"].sum()
    
    all_matches = pd.concat([
        home_matches.assign(goals_for=home_matches["home_score"], goals_against=home_matches["away_score"], won=(home_matches["home_score"] > home_matches["away_score"]).astype(int)),
        away_matches.assign(goals_for=away_matches["away_score"], goals_against=away_matches["home_score"], won=(away_matches["away_score"] > away_matches["home_score"]).astype(int))
    ]).sort_values("date")
    
    recent_10 = all_matches.tail(10)
    if len(recent_10) < 10:
        recent_form_val = 0.5
    else:
        recent_form_val = recent_10["won"].mean()
    
    team_stats[team] = {
        "winrate": float(total_wins / total_matches),
        "goal_avg": float(total_goals / total_matches),
        "recent_form": float(recent_form_val),
        "matches_played": int(total_matches)
    }

joblib.dump(model, "models/match_predictor.pkl")
joblib.dump({"team_stats": team_stats, "feature_cols": feature_cols}, "models/team_data.pkl")

print(f"Teams stored: {len(team_stats)}")

high_match_teams = {k: v for k, v in team_stats.items() if v["matches_played"] >= 100}
top_5 = sorted(high_match_teams.items(), key=lambda x: x[1]["winrate"], reverse=True)[:5]
for team, stats in top_5:
    print(f"{team}: {stats['winrate']:.3f}")

# Task 8: Try the model

In [ ]:
def predict_match(team_a, team_b, is_neutral=True, is_major_tournament=True):
    if team_a not in team_stats:
        raise ValueError(f"Team '{team_a}' not found in team_stats. Please use a team that has participated in Soccer World Cup qualification.")
    if team_b not in team_stats:
        raise ValueError(f"Team '{team_b}' not found in team_stats. Please use a team that has participated in Soccer World Cup qualification.")
    
    stats_a = team_stats[team_a]
    stats_b = team_stats[team_b]
    
    row_data = {
        "team_a_winrate": stats_a["winrate"],
        "team_b_winrate": stats_b["winrate"],
        "team_a_goal_avg": stats_a["goal_avg"],
        "team_b_goal_avg": stats_b["goal_avg"],
        "team_a_recent_form": stats_a["recent_form"],
        "team_b_recent_form": stats_b["recent_form"],
        "is_neutral": int(is_neutral),
        "is_major_tournament": int(is_major_tournament)
    }
    
    row_df = pd.DataFrame([row_data])
    row_df = row_df[feature_cols]
    
    proba = model.predict_proba(row_df)
    
    return {
        "team_a_win_prob": float(proba[0][0]),
        "draw_prob": float(proba[0][1]),
        "team_b_win_prob": float(proba[0][2])
    }

print(predict_match("Brazil", "Argentina"))
print(predict_match("Germany", "Brazil"))

# Task 9: Create an interactive UI application

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
from pathlib import Path

st.set_page_config(page_title="Soccer 2026 Match Predictor", page_icon="⚽", layout="centered")

@st.cache_resource
def load_artifacts():
    model = joblib.load("models/match_predictor.pkl")
    team_data = joblib.load("models/team_data.pkl")
    team_stats = team_data["team_stats"]
    feature_cols = team_data["feature_cols"]
    return model, team_stats, feature_cols

model, team_stats, feature_cols = load_artifacts()

st.title("⚽ Soccer 2026 Match Predictor")
st.caption("Predictions based on historical international football results")

team_names = sorted(team_stats.keys())

col1, col2 = st.columns(2)
with col1:
    default_a = team_names.index("Brazil") if "Brazil" in team_names else 0
    team_a = st.selectbox("Team A", team_names, index=default_a)
with col2:
    default_b = team_names.index("Argentina") if "Argentina" in team_names else 1
    team_b = st.selectbox("Team B", team_names, index=default_b)

is_neutral = st.checkbox("Neutral venue", value=True)
is_major_tournament = st.checkbox("Major tournament (e.g. World Cup)", value=True)

if st.button("Predict", type="primary", use_container_width=True):
    if team_a == team_b:
        st.error("Please pick two different teams.")
    else:
        stats_a = team_stats[team_a]
        stats_b = team_stats[team_b]
        
        row_data = {
            "team_a_winrate": stats_a["winrate"],
            "team_b_winrate": stats_b["winrate"],
            "team_a_goal_avg": stats_a["goal_avg"],
            "team_b_goal_avg": stats_b["goal_avg"],
            "team_a_recent_form": stats_a["recent_form"],
            "team_b_recent_form": stats_b["recent_form"],
            "is_neutral": int(is_neutral),
            "is_major_tournament": int(is_major_tournament)
        }
        
        row_df = pd.DataFrame([row_data])
        row_df = row_df[feature_cols]
        
        proba = model.predict_proba(row_df)
        
        team_a_prob = proba[0][0]
        draw_prob = proba[0][1]
        team_b_prob = proba[0][2]
        
        col1, col2, col3 = st.columns(3)
        with col1:
            st.metric(f"{team_a} wins", f"{team_a_prob * 100:.1f}%")
        with col2:
            st.metric("Draw", f"{draw_prob * 100:.1f}%")
        with col3:
            st.metric(f"{team_b} wins", f"{team_b_prob * 100:.1f}%")
        
        st.progress(team_a_prob, text=f"{team_a} wins")
        st.progress(draw_prob, text="Draw")
        st.progress(team_b_prob, text=f"{team_b} wins")
        
        comparison_data = {
            "Metric": ["Win rate", "Avg goals scored", "Recent form (last 10)", "Matches played"],
            team_a: [
                f"{stats_a['winrate']:.3f}",
                f"{stats_a['goal_avg']:.2f}",
                f"{stats_a['recent_form']:.3f}",
                stats_a['matches_played']
            ],
            team_b: [
                f"{stats_b['winrate']:.3f}",
                f"{stats_b['goal_avg']:.2f}",
                f"{stats_b['recent_form']:.3f}",
                stats_b['matches_played']
            ]
        }
        st.table(pd.DataFrame(comparison_data))

# Task 10: Launch the UI application

In [2]:
import subprocess
import sys
import time
import webbrowser

streamlit_process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.headless", "true", "--server.port", "8501", "--browser.gatherUsageStats", "false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("Starting Streamlit server...")
time.sleep(4)

webbrowser.open("http://localhost:8501")

print("Streamlit app is running at http://localhost:8501")
print("To stop the server later, run: streamlit_process.terminate()")

Starting Streamlit server...
Streamlit app is running at http://localhost:8501
To stop the server later, run: streamlit_process.terminate()
